## Reader and Writer Monads

In part 3 of this series, we finally tackled the conceptual idea of monads. We learned what they are, and saw how some common types like IO and Maybe work as monads. Now in this part, we'll start looking at some other useful monads. In particular, we'll consider the Reader and Writer monads.

If you haven't started writing your own Haskell yet, you have all the tools and skills you need to do so! Just download our Beginners Checklist to learn how to get started! Writing your own code will help a lot with understanding the examples in these articles!

This article has quite a bit of code that goes along with it. In the Github Repository for this series, you can follow along with it! You can take a look at the ReaderWriter module and fill in the "TODOs" as you're reading this article. If you're just looking for the final reference, you can also open up ReaderWriteComplete.

## Global Variables (or a lack thereof)
In Haskell, our code is generally "pure", meaning functions can only interact with the arguments passed to them. This effectively means we cannot have global variables. We can have global expressions, but these are fixed at compile time. If user behavior might change them, we have to wrap them in the IO monad, which means they can't be used from pure code.

Consider this example. Here, we want to have an Environment containing different parameters as a global variable. However, we might have to load these from a config file or a command line interface, which requires the IO monad.

```
main1 :: IO ()

main1 = do
  env <- loadEnv
  let str = func1 env
  print str

data Environment = Environment
  { param1 :: String
  , param2 :: String
  , param3 :: String
  }

loadEnv :: IO Environment
loadEnv = ...

func1 :: Environment -> String
func1 env = "Result: " ++ (show (func2 env))

func2 :: Environment -> Int
func2 env = 2 + floor (func3 env)

func3 :: Environment -> Float
func3 env = (fromIntegral $ l1 + l2 + l3) * 2.1
  where
    l1 = length (param1 env)
    l2 = length (param2 env) * 2
    l3 = length (param3 env) * 3
```



The only function actually using the environment is func3. However func3 is a pure function. This means it cannot directly call loadEnv, an impure function in the IO monad. This means the environment has to be passed through as a variable to the other functions, just so they can ultimately pass it to func3. In a language with global variables, we could save env as a global value in main. Then func3 could access it directly. There would be no need to have it as a parameter to func1 and func2. In larger programs, these "pass-through" variables can cause a lot of headaches.

## The Reader Solution
The Reader monad solves this problem. It effectively creates a global read-only value of a specified type. All functions within the monad can "read" the type. Let's look at how the Reader monad changes the shape of our code. Our functions no longer need the Environment as an explicit parameter, as they can access it through the monad.

```
main :: IO ()
main = do
  env <- loadEnv
  let str = runReader func1' env
  print str

func1' :: Reader Environment String
func1' = do
  res <- func2'
  return ("Result: " ++ show res)

func2' :: Reader Environment Int
func2' = do
  env <- ask
  let res3 = func3 env
  return (2 + floor res3)

-- as above
func3 :: Environment -> Float
...
```

The ask function unwraps the environment so we can use it. The monad's bind action allows us to glue different Reader actions together together. In order to call a reader action from pure code, all we need to do is call the runReader function and supply the environment as a parameter. All functions within the action will be able to treat it like a global variable.

The above code also introduces an important idea. Whenever you learn about a monad "X", there's often a corresponding function "runX" that tells you how to run operations of that monad from a pure context (IO is an exception). This function will often require some kind of input, as well as the computation itself. Then it will produce the final output of the computation. In the case of Reader, we have the runReader function. It requires the value we'll use to read from, as well as the Reader computation.

```
runReader :: Reader r a -> r -> a
```

It might not seem like we've accomplished much, but our code is much more intuitive now. We keep func3 as it was. It makes sense to describe it as a function from an Environment to a value. However, our other two functions no longer take the environment as an explicit parameter. They simply exist in a context where the environment is a global variable.

## Accumulating Values

Now, to motivate the Writer monad, let's talk about the accumulation problem. Suppose we have a few different functions. Each will perform some string operations we've assigned an arbitrary "cost" to. We want to keep track of how "expensive" it was to run the full computation. We can do this by using accumulator arguments to keep track of the cost we've seen so far. We then keep passing the accumulated value along with the final String result.

In [3]:
-- Calls func2 if even length, func3 and func4 if odd
func1 :: String -> (Int, String)
func1 input = if length input `mod` 2 == 0
  then func2 (0, input)
  else (i1 + i2, str1 ++ str2)
    where
      (i1, str1) = func3 (0, tail input)
      (i2, str2) = func4 (0, take 1 input)

-- Calls func4 on truncated version
func2 :: (Int, String) -> (Int, String)
func2 (prev, input) = if (length input) > 10
  then func4 (prev + 1, take 9 input)
  else (10, input)

-- Calls func2 on expanded version if a multiple of 3
func3 :: (Int, String) -> (Int, String)
func3 (prev, input) = if (length input) `mod` 3 == 0
  then (prev + f2resI + 3, f2resStr)
  else (prev + 1, tail input)
  where
    (f2resI, f2resStr) = func2 (prev, input ++ "ab")

func4 :: (Int, String) -> (Int, String)
func4 (prev, input) = if (length input) < 10
  then (prev + length input, input ++ input)
  else (prev + 5, take 5 input)

Line 3: Use even
Found:
length input `mod` 2 == 0
Why not:
even (length input)Line 12: Redundant bracket
Found:
(length input) > 10
Why not:
length input > 10Line 18: Redundant bracket
Found:
(length input) `mod` 3
Why not:
length input `mod` 3Line 25: Redundant bracket
Found:
(length input) < 10
Why not:
length input < 10

First of all, we can notice that this function structure is a little bit cumbersome. Once again, we're passing around extra parameters. In particular, we're tracking the accumulated cost, which shows up as an input and an output to each function. The Writer monad provides us with an easier way to track this value. It would also make it easier for us to represent the cost with a different type. But to understand how, we should first learn two typeclasses, Semigroup and Monoid, that help us generalize accumulation.

In [ ]:
import Control.Monad.Writer

acc2' :: String -> Writer Int String
acc2' input = if (length input) > 10
  then do
    tell 1
    acc4' (take 9 input)
  else do
    tell 10
    return input

Line 6: Redundant bracket
Found:
(length input) < 10
Why not:
length input < 10

We branch on the length of the input, and then each branch is a "do" statement. We'll use tell to provide the appropriate value to increment the accumulator, and then move on and call the next function, or return our answer. Then acc3 and acc4 are similar.

In [10]:
acc3' :: String -> Writer Int String
acc3' input = if (length input) `mod` 3 == 0
  then do
    tell 3
    acc2' (input ++ "ab")
  else do
    tell 1
    return $ tail input

acc4' :: String -> Writer Int String
acc4' input = if (length input) < 10
  then do
    tell (length input)
    return (input ++ input)
  else do
    tell 5
    return (take 5 input)

Line 2: Redundant bracket
Found:
(length input) `mod` 3
Why not:
length input `mod` 3Line 11: Redundant bracket
Found:
(length input) < 10
Why not:
length input < 10

: 

Finally, we don't change the type signature of our original function, but we instead use runWriter to call our helpers, as appropriate.

In [11]:
acc1' :: String -> (String, Int)
acc1' input = if length input `mod` 2 == 0
  then runWriter (acc2' input)
  else runWriter $ do
    str1 <- acc3' (tail input)
    str2 <- acc4' (take 1 input)
    return (str1 ++ str2)

Line 2: Use even
Found:
length input `mod` 2 == 0
Why not:
even (length input)

: 

Notice we no longer need to actually explicitly keep track of the accumulator. It is now wrapped by the Writer monad. We can increase it in any of our functions by calling "tell". Now our code is much simpler and our types are cleaner.

## Conclusion

Now that we know about the Reader and Writer monads, it's time to move on to part 5. There, we'll discuss the State monad. This monad combines these two concepts into a read/write state, essentially allowing the full privilege of a global variable. If these concepts were still a little confusing, don't be afraid to take another look at part 3 to solidify your understanding of monads.

Maybe you've never programmed in Haskell before, and the realization that we can get all this cool functionality makes you want to try it! Download our Beginners Checklist to get started!

In [ ]:
import Control.Monad.Writer
import Data.Monoid (Sum(..)) -- Importa o construtor Sum para podermos somar

-- 1. Alteramos a assinatura de todos de 'Writer Int String' para 'Writer (Sum Int) String'
acc4' :: String -> Writer (Sum Int) String
acc4' input = if (length input) < 10
  then do
    tell (Sum (length input)) -- Embrulha o Int em Sum
    return (input ++ input)
  else do
    tell (Sum 5)
    return (take 5 input)

acc2' :: String -> Writer (Sum Int) String
acc2' input = if (length input) > 10
  then do
    tell (Sum 1)
    acc4' (take 9 input)
  else do
    tell (Sum 10)
    return input

acc3' :: String -> Writer (Sum Int) String
acc3' input = if (length input) `mod` 3 == 0
  then do
    tell (Sum 3)
    acc2' (input ++ "ab")
  else do
    tell (Sum 1)
    return $ tail input

-- 2. Na função principal, usamos o padrão (Sum resultado) para extrair o Int puro no final
acc1' :: String -> (String, Int)
acc1' input =
  let (str, Sum total) = if length input `mod` 2 == 0
                           then runWriter (acc2' input)
                           else runWriter $ do
                             str1 <- acc3' (tail input)
                             str2 <- acc4' (take 1 input)
                             return (str1 ++ str2)
  in (str, total)

Line 6: Redundant bracket
Found:
(length input) < 10
Why not:
length input < 10Line 15: Redundant bracket
Found:
(length input) > 10
Why not:
length input > 10Line 24: Redundant bracket
Found:
(length input) `mod` 3
Why not:
length input `mod` 3Line 35: Use even
Found:
length input `mod` 2 == 0
Why not:
even (length input)

In [14]:
-- Testando com string de tamanho PAR (ex: "Haskell" tem 7, mas "Haskell!" tem 8)
print $ acc1' "Haskell!"

-- Testando com string de tamanho ÍMPAR (ex: "Hello" tem 5)
print $ acc1' "Hello"


("Haskell!",10)

("lloHH",2)